# MODELO LSTM PARA EL CONTAMINANTE PM10 PARA BARCELONA


In [13]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from pathlib import Path
from sklearn.model_selection import ParameterGrid

import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("fbprophet").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
from pylab import rcParams
plt.style.use("fivethirtyeight")
plt.rcParams["lines.linewidth"] = 1.5
light_style = {
    "figure.facecolor": "#d9effb",   
    "axes.facecolor": "#d9effb",
    "savefig.facecolor": "#d9effb",
    "axes.grid": True,
    "axes.grid.which": "both",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "grid.color": "#a9d3f2",
    "grid.linewidth": "0.8",
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    "axes.labelweight": "black",      
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 12,
    "axes.titleweight": "bold",       
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
}
plt.rcParams.update(light_style)
rcParams["figure.figsize"] = (18, 7)

import sys
import importlib
from pathlib import Path

SCRIPTS_PATH = Path.cwd().parents[2]

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.append(str(SCRIPTS_PATH))

import utils
importlib.reload(utils)
from utils import CARGA_Y_FILTRO, EVALUAR_METRICAS, ESCALADOR, BUSQUEDA_CONFIGURACIONES_LSMT, ENTRENAR_EVALUAR_LSTM

In [14]:
BASE_PATH = Path("..", "..", "..", "..")
FOLDER_DATA = BASE_PATH / "datasets" / "eda_archivos_cont_clima_indices"

## Carga de los datos y división del conjunto de datos

Cargamos los datos y filtramos las columnas que realmente necesitamos y como ciudad elegimos únicamente Barcelona.

In [15]:
df1 = CARGA_Y_FILTRO(
    contaminante="PM10 (ug.m-3)",
    ciudad="Barcelona"
)

In [16]:
# ==============================================================================
# División cronológica del conjunto de datos
# ==============================================================================

train = df1[df1["Año"] <= 2022].copy()

validation = df1[
    (df1["Año"] >= 2023) &
    (df1["Año"] <= 2023)
].copy()

test = df1[df1["Año"] >= 2024].copy()

print(f"Entrenamiento: {train['Start'].min()} -> {train['Start'].max()}")
print(f"Validación:    {validation['Start'].min()} -> {validation['Start'].max()}")
print(f"Prueba:        {test['Start'].min()} -> {test['Start'].max()}")

print()
print(f"Nº muestras entrenamiento: {len(train):,}")
print(f"Nº muestras validación:    {len(validation):,}")
print(f"Nº muestras prueba:        {len(test):,}")

Entrenamiento: 2017-01-01 01:00:00 -> 2022-12-31 23:00:00
Validación:    2023-01-01 00:00:00 -> 2023-12-31 23:00:00
Prueba:        2024-01-01 00:00:00 -> 2024-12-31 23:00:00

Nº muestras entrenamiento: 52,583
Nº muestras validación:    8,760
Nº muestras prueba:        8,784


Para seguir con la forma en que Prophet denominaba a la variable objetivo y la columna temporal, renombramos las fechas por *ds* t la variable objetivo por *y*.

In [17]:
train = train.rename(columns={"Start": "ds", "PM10 (ug.m-3)": "y"})
validation = validation.rename(columns={"Start": "ds", "PM10 (ug.m-3)": "y"})
test = test.rename(columns={"Start": "ds", "PM10 (ug.m-3)": "y"})

In [18]:
variables_exogenas= [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloudcover",
    "windspeed_10m",
    "shortwave_radiation",
    "boundary_layer_height",
    "NDVI",
    "NDBI"
]

Nos quedamos exclusivamente con las variables necesarias para el modelado.

In [19]:
# Lista de columnas que quieres mantener
columnas = ['ds', 'y'] + variables_exogenas

train = train[columnas]
validation = validation[columnas]
test = test[columnas]

## Escalado de los datos

In [20]:
(
    train_lstm,
    validation_lstm,
    test_lstm,
    scaler_y,
    scaler_X
) = ESCALADOR(
    train=train,
    validation=validation,
    test=test,
    variables_exogenas=variables_exogenas
)

## Selección de los mejores hiperparámetros

In [21]:
param_grid = {

    # Número de neuronas de las capas LSTM
    "units": [32, 64],

    # Número de capas LSTM
    "num_layers": [1, 2],

    # Dropout
    "dropout": [0.0, 0.2],

    # Optimizador
    "optimizer": ["adam", "rmsprop"],

    # Número de épocas
    "epochs": [50, 100],

    # Tamaño del batch
    "batch_size": [32, 64],

    # Número de horas anteriores consideradas
    "input_size": [24, 48, 72, 168]
}

In [22]:
# ==============================================================================
# Generación de todas las combinaciones posibles
# ==============================================================================

configuraciones = list(
    ParameterGrid(param_grid)
)

print(
    f"Número total de configuraciones que se evaluarán: "
    f"{len(configuraciones)}"
)

Número total de configuraciones que se evaluarán: 256


In [ ]:
resultados_lstm, mejor_configuracion_lstm = BUSQUEDA_CONFIGURACIONES_LSMT(
    train=train_lstm,
    validation=validation_lstm,
    variables_exogenas=variables_exogenas,
    param_grid=param_grid,
    scaler_y=scaler_y,
    seed=42
)


Mejores parámetros:

units: 64
num_layers: 1
dropout: 0.2
optimizer: adam
epochs: 100
batch_size: 64
input_size: 168

MSE de validación: 137.204061


In [24]:
# Mejores parámetros obtenidos tras la búsqueda de hiperparámetros

mejores_parametros = {
    "units": 64,
    "num_layers": 1,
    "dropout": 0.2,
    "optimizer": "adam",
    "epochs": 100,
    "batch_size": 64,
    "input_size": 168
}

In [ ]:
modelo_lstm_final, metricas_lstm, y_test_real, y_test_predicho, fechas_test = (
    ENTRENAR_EVALUAR_LSTM(
        train=train_lstm,
        validation=validation_lstm,
        test=test_lstm,
        variables_exogenas=variables_exogenas,
        mejores_parametros=mejores_parametros,
        scaler_y=scaler_y,
        seed=42
    )
)

Resultados de la evaluación del modelo
--------------------------------------
Error absoluto medio (MAE): 10.142496
Error cuadrático medio (MSE): 137.204061
Raíz del error cuadrático medio (RMSE): 11.713414
Error porcentual absoluto medio (MAPE): 19.46 %
Raíz del error cuadrático medio normalizada (NRMSE): 53.81 %
